# Q&A notebook

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
plt.rcParams['svg.fonttype'] = 'none'
from yaml import load, Loader
from daforfer import DaforferDB
conf = load(open("conf.yaml"), Loader)
db = DaforferDB(conf['database'])
db.toc()

┌──────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│         name         │                                                                                  description                                                                                   │
│       varchar        │                                                                                    varchar                                                                                     │
├──────────────────────┼────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ D_sites              │ This table contains key information about each of the libraries, such as their site, habitat and host                                                                  

## How many virus detections were reported in McLeish2024?

In [2]:

len(db.conn.sql('SELECT * FROM D_virusHits').df()) # 1616 hits in our database q000016

1616

## How many taxa?

In [3]:
len(db.conn.sql('SELECT * FROM D_virusHits').df()['scientific_name'].unique()) # 158 hits in our database q000017

158

## Did all the PAB bacteria detected correspond to at least a virus in their library?

In [4]:
virus = db.conn.sql('SELECT * FROM D_virusHits').df()
bacteria = db.conn.sql('SELECT * FROM D_pabHits').df()
virus_positive_libraries = virus['library'].unique()
bacteria['is_in_virus'] = bacteria['library'].isin(virus_positive_libraries)
bacteria.groupby(['scientific_name'], as_index=False)['is_in_virus'].sum().query('is_in_virus != 0')

,scientific_name,is_in_virus
0,Achromobacter xylosoxidans,2
1,Acidovorax sp. Leaf160,1
2,Acidovorax sp. Leaf78,1
3,Acinetobacter baumannii,1
4,Acinetobacter pittii,3
...,...,...
122,Stutzerimonas stutzeri,1
123,Variovorax paradoxus,2
124,Xanthomonas campestris,6
125,Xaviernesmea rhizosphaerae,1


In [5]:
len(bacteria.taxid.unique())

127

## Were all the virus detected in the pressence of a bacteria?

In [6]:
virus = db.conn.sql('SELECT * FROM D_virusHits').df()
bacteria = db.conn.sql('SELECT * FROM D_pabHits').df()
bacteria_positive_libraries = bacteria['library'].unique()
virus['is_in_bacteria'] = virus['library'].isin(bacteria_positive_libraries)
virus.groupby(['scientific_name'], as_index=False)['is_in_bacteria'].sum().query('is_in_bacteria != 0')

,scientific_name,is_in_bacteria
0,African eggplant yellowing virus,2
1,Ageratum latent virus RNA 1,1
2,Alfalfa enamovirus 1,1
3,Alfalfa mosaic virus RNA 2,3
4,Arabidopsis halleri partitivirus 1,1
...,...,...
147,Watermelon mosaic virus,26
153,Yam bean mosaic virus,2
154,Yam mosaic virus,4
156,Youcai mosaic virus,17


## How many libraries has the site with most libraries?

In [7]:
pd.read_csv('output/metadata.site-library.csv', sep=';').value_counts('site')

site
L1    40
L3    35
L2    35
Q2    25
Q1    24
L4    23
E2    21
E4    19
Q3    16
E3    15
E1    14
Q4    13
M1     8
Z1     6
C2     6
C1     5
M3     5
H3     5
M4     4
M2     4
H2     4
H1     4
Z2     4
Name: count, dtype: int64

## What's the virus with wider range?

In [8]:
m = pd.merge(
    pd.read_csv("output/hits.virus.csv", sep=';'),
    pd.read_csv('output/metadata.site-library.csv', sep=';'),
    on='library'
)[['acronym', 'taxid', 'host_taxon']].drop_duplicates().value_counts('acronym').reset_index()

In [9]:
m

,acronym,count
0,CMV3,83
1,TMGMV,65
2,PZSV3,58
3,TMV,52
4,RuCMV,50
...,...,...
153,SbDV,1
154,CymRSV,1
155,CaMV,1
156,PCV2,1


In [10]:
m.query('count == 1')

,acronym,count
95,TYMV,1
96,BPeMV,1
97,BYSMV,1
98,BYDV_GPV,1
99,BYDV_KerII,1
...,...,...
153,SbDV,1
154,CymRSV,1
155,CaMV,1
156,PCV2,1


In [11]:
m = pd.merge(
    pd.read_csv("output/hits.bacteria.csv", sep=';').query('is_pab == True'),
    pd.read_csv('output/metadata.site-library.csv', sep=';'),
    on='library'
)[['scientific_name', 'taxid', 'host_taxon']].drop_duplicates().value_counts('scientific_name').reset_index()

In [12]:
len(m.query('count == 1'))

55

In [13]:
db.conn.close()